In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline

In [ ]:
ROOT = Path.cwd().parent

# Load data
df = pd.read_excel(ROOT / 'data' / 'raw' / 'Telco_customer_churn.xlsx')

df.head()

### Data Cleaning

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.duplicated().sum()

In [ ]:
df['CustomerID'].duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:
df[df['Churn Reason'].isnull()]['Churn Label'].value_counts()

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
for col in df.select_dtypes(include='object').columns:
    print(f"\n{col}:")
    print(df[col].value_counts())

In [ ]:
# Dropping columns that are non-informative or irrelevant for EDA and modeling:
# - Count: constant value of 1 across all 7043 rows, adds no information
# - Country: single value 'United States' across all rows
# - State: single value 'California' across all rows
# - Zip Code: too granular, high cardinality, not useful for EDA or modeling
# - Lat Long: combined string of Latitude and Longitude, redundant
# - Latitude & Longitude: geographic coordinates, not relevant for EDA or ML modeling

df.drop(columns=['Count', 'Country', 'State', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude'], inplace=True)

In [ ]:
# Save processed file
df.to_csv(ROOT / 'data' / 'processed' / 'Telco_customer_churn_processed.csv', index=False)

## Exploratory Data Analysis (EDA)

In [ ]:
df.columns

### Churn Analysis

In [ ]:
sns.boxplot(data=df, x='Churn Label', y='Monthly Charges')
plt.title('Monthly Charges Distribution by Churn Status')
plt.show()

In [ ]:
df[df['Churn Label'] == 'Yes']['Monthly Charges'].max()

In [ ]:
sns.boxplot(data=df, x='Churn Label', y='Tenure Months')
plt.title('Tenure Months by Churn Status')
plt.show()

In [ ]:
# Total Charges contained 11 blank space values (" ") not detected during initial null check
# Discovered during EDA when attempting boxplot visualization
# Converted to numeric and filled with 0 as these correspond to Tenure Month = 0 customers

df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'] = df['Total Charges'].fillna(0)

In [ ]:
sns.boxplot(data=df, x='Churn Label', y='Total Charges')
plt.title('Total Charges by Churn Status')
plt.show()

Hoppers pay high monthly rates but leave early — resulting in low total revenue contribution despite being high monthly charge customers

In [ ]:
sns.boxplot(data=df, x='Churn Label', y='CLTV')
plt.title('Customer Lifetime Value by Churn Status')
plt.show()

In [ ]:
sns.countplot(data=df, x='Churn Value')
plt.title('Churn distribution')
plt.show()

In [ ]:
sns.countplot(data=df, x='Contract', hue='Churn Label')
plt.title('Churn Distribution by Contract Type')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(data=df, x='Payment Method', hue='Churn Label', ax=axes[0])
axes[0].set_title('Churn Count by Payment Method')
axes[0].tick_params(axis='x', rotation=45)

# Churn rate plot
churn_rate = df.groupby('Payment Method')['Churn Value'].mean().reset_index()
churn_rate.columns = ['Payment Method', 'Churn Rate']
sns.barplot(data=churn_rate, x='Payment Method', y='Churn Rate', ax=axes[1])
axes[1].set_title('Churn Rate by Payment Method')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
sns.countplot(data=df, x='Internet Service', hue='Churn Label')
plt.title('Churn Distribution by Internet Sevice')
plt.show()

Fiber optic customers on month-to-month contracts paying via electronic check are your highest churn risk

In [ ]:
sns.countplot(data=df, x='Gender', hue='Churn Label')
plt.title('Churn Distribution by Gender')
plt.show()

Gender has no significant impact on churn — churn is evenly distributed across male and female customers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
sns.countplot(data=df, x='Senior Citizen', hue='Churn Label', ax=axes[0])
axes[0].set_title('Churn Count by Senior Citizen')

# Churn rate plot
churn_rate = df.groupby('Senior Citizen')['Churn Value'].mean().reset_index()
churn_rate.columns = ['Senior Citizen', 'Churn Rate']
sns.barplot(data=churn_rate, x='Senior Citizen', y='Churn Rate', ax=axes[1])
axes[1].set_title('Churn Rate by Senior Citizen')

plt.tight_layout()
plt.show()

Senior citizens are a small portion of customers but churn at nearly double the rate of non-senior citizens

In [ ]:
sns.countplot(data=df, x='Partner', hue='Churn Label')
plt.title('Churn Distribution by Partner Status')
plt.show()

In [ ]:
sns.countplot(data=df, x='Dependents', hue='Churn Label')
plt.title('Churn Distribution by Dependents')
plt.show()

In [ ]:
sns.countplot(data=df, x='Paperless Billing', hue='Churn Label')
plt.title('Churn Distribution by Paperless Billing')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(data=df, x='Phone Service', hue='Churn Label', ax=axes[0])
axes[0].set_title('Churn Count by Phone Service')

churn_rate = df.groupby('Phone Service')['Churn Value'].mean().reset_index()
churn_rate.columns = ['Phone Service', 'Churn Rate']
sns.barplot(data=churn_rate, x='Phone Service', y='Churn Rate', ax=axes[1])
axes[1].set_title('Churn Rate by Phone Service')

plt.tight_layout()
plt.show()

In [ ]:
sns.countplot(data=df, x='Multiple Lines', hue='Churn Label')
plt.title('Churn Distribution by Multiple Lines')
plt.show()

In [ ]:
sns.countplot(data=df, x='Online Security', hue='Churn Label')
plt.title('Churn Distribution by Online Security')
plt.show()

In [ ]:
sns.countplot(data=df, x='Tech Support', hue='Churn Label')
plt.title('Churn Distribution by Tech Support')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.select_dtypes(include='number').corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
df['Churn Reason'].value_counts().head(10).plot(kind='barh')
plt.title('Top 10 Churn Reasons')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

## EDA Summary — Churn Analysis

### Dataset Overview
- Total customers: 7,043 | Churned: ~1,869 (26%) | Retained: ~5,174 (74%)
- Class imbalance is moderate and will be addressed during modeling

### Key Churn Drivers

**Contract Type**
- Month-to-month customers have the highest churn count by far
- One and two year contract customers rarely churn due to long term commitment

**Monthly Charges**
- Churned customers have a higher median monthly charge (~79) vs retained (~62)
- Higher charges on flexible contracts create strong incentive to switch providers

**Tenure Months**
- Churned customers have significantly lower tenure — newer customers are more likely to leave
- Long term customers build loyalty and rarely churn (confirmed by -0.35 correlation with Churn Value)

**Internet Service**
- Fiber optic customers churn the most — premium service with highest monthly charges
- Customers with no internet service barely churn — low cost, low dissatisfaction

**Online Security & Tech Support**
- Customers without online security churn at ~41% vs only ~15% for those with it
- Customers without tech support churn at ~41% vs ~15% for those with it
- Customers who invest in add-on services are significantly more committed and loyal
- Lack of support and security features indicates bare minimum plans — easier to leave
- Both are strong churn predictors and will be included in modeling

**Payment Method**
- Electronic check users have the highest churn rate
- Automatic payment methods (credit card, bank transfer) correlate with lower churn — more committed customers

**Paperless Billing**
- Paperless billing customers churn more — digitally engaged and price aware, more likely to compare and switch

**Demographics**
- Gender has no impact on churn — evenly distributed across male and female
- Senior citizens churn at nearly double the rate (~45%) despite being a small portion of customers
- Customers without a partner or dependents churn significantly more — less financial commitment and easier to switch

**Total Charges & CLTV**
- Churned customers have low total charges despite high monthly charges — confirming short tenure
- CLTV shows marginal difference and is a derived metric — excluded from modeling to avoid leakage

### High Risk Customer Profile
> "A new fiber optic customer on a month-to-month contract, paying via electronic check,
> enrolled in paperless billing, with no partner or dependents — is your highest churn risk"

### Features Dropped Before Modeling
- Gender: no predictive value
- Churn Score, CLTV, Churn Reason, Churn Label: data leakage or redundancy

### Correlation Highlights
- Tenure Months & Total Charges: strong positive (0.83) — multicollinearity risk
- Churn Score & Churn Value: strong positive (0.66) — confirms leakage
- Tenure Months & Churn Value: negative (-0.35) — longer tenure = lower churn risk